In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets,transforms

In [2]:
# Use the dataset in torchvision:
train_dataset=datasets.FashionMNIST(root='/dataset',train=True,transform=transforms.ToTensor(),download=True)
test_dataset=datasets.FashionMNIST(root='/dataset',train=False,transform=transforms.ToTensor())


In [3]:
train_dataset

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: /dataset
    Split: Train
    StandardTransform
Transform: ToTensor()

In [4]:
test_dataset

Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: /dataset
    Split: Test
    StandardTransform
Transform: ToTensor()

In [5]:
# Define the architecture:
class LogisticRegression(nn.Module):
  def __init__(self,n_dim,n_classes):
    super().__init__()
    self.linear=nn.Linear(n_dim,n_classes)

  def forward(self,x):
    out=self.linear(x)
    return out

In [6]:
# Create the data loaders to send it into batches: (Why shuffle only train and not test?)
train_dataloader=DataLoader(train_dataset,shuffle=True,batch_size=64)
test_dataloader=DataLoader(test_dataset,shuffle=False,batch_size=64)

In [7]:
len(train_dataset)

60000

In [8]:
len(train_dataloader)

938

In [9]:
# Define the loss function and optimizers:
model=LogisticRegression(28*28,10)
loss=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-4)
epochs=10

In [10]:
# Start the training loop:
for epoch in range(epochs):
  avg_loss=0
  train_correct=0
  total_count=0

  model.train()
  for batch_feature,batch_label in train_dataloader:
    optimizer.zero_grad()

    img=batch_feature.view(batch_feature.size(0),-1)
    out=model(img)

    model_loss=loss(out,batch_label)
    model_loss.backward()
    optimizer.step()

    avg_loss+=model_loss.item()

    _,pred=torch.max(out,1)
    train_correct+=(pred==batch_label).sum().item()
    total_count+=batch_feature.size(0)

  model.eval()
  test_avg_loss=0
  test_correct=0
  test_total_count=0

  with torch.no_grad():
    for batch_feature,batch_label in test_dataloader:
      img=batch_feature.view(batch_feature.size(0),-1)
      out=model(img)

      model_loss=loss(out,batch_label)
      test_avg_loss+=model_loss.item()


      _,pred=torch.max(out,1)
      test_correct+=(pred==batch_label).sum().item()
      test_total_count+=batch_feature.size(0)

    print(f"Epochs: {epoch+1} | Train Loss: {avg_loss/len(train_dataloader):.4f} | Test Loss:{test_avg_loss/len(test_dataloader):.4f} | Train Accuracy:{train_correct/total_count:.4f} | Test Accuracy:{test_correct/test_total_count:.4f}")



Epochs: 1 | Train Loss: 1.1914 | Test Loss:0.8518 | Train Accuracy:0.6556 | Test Accuracy:0.7144
Epochs: 2 | Train Loss: 0.7501 | Test Loss:0.7032 | Train Accuracy:0.7572 | Test Accuracy:0.7638
Epochs: 3 | Train Loss: 0.6495 | Test Loss:0.6373 | Train Accuracy:0.7897 | Test Accuracy:0.7865
Epochs: 4 | Train Loss: 0.5967 | Test Loss:0.5980 | Train Accuracy:0.8044 | Test Accuracy:0.7999
Epochs: 5 | Train Loss: 0.5628 | Test Loss:0.5710 | Train Accuracy:0.8150 | Test Accuracy:0.8099
Epochs: 6 | Train Loss: 0.5389 | Test Loss:0.5539 | Train Accuracy:0.8218 | Test Accuracy:0.8124
Epochs: 7 | Train Loss: 0.5210 | Test Loss:0.5381 | Train Accuracy:0.8273 | Test Accuracy:0.8188
Epochs: 8 | Train Loss: 0.5071 | Test Loss:0.5266 | Train Accuracy:0.8321 | Test Accuracy:0.8204
Epochs: 9 | Train Loss: 0.4958 | Test Loss:0.5179 | Train Accuracy:0.8351 | Test Accuracy:0.8244
Epochs: 10 | Train Loss: 0.4866 | Test Loss:0.5112 | Train Accuracy:0.8384 | Test Accuracy:0.8261
